# 迷宫
给定不全的迷宫图，预测迷宫的特征: 
- 强连通分量
- 最短路
-  
-  

## 解法
手动提取特征，然后用random_forest回归

In [ ]:
ONLINE = True

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import random
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import train_test_split
import zipfile
import os
from scipy.ndimage import label
import networkx

seed = 42
USE_VALIDATION = False  # 设置为 True 使用验证集，设置为 False 使用全量训练

random.seed(seed)  # Python built-in random
np.random.seed(seed)  # NumPy

N = 30
D = N * N

# ===== 数据读取与编码 =====
def encode_lines(path: str) -> np.ndarray:
    char_id = {".": 0, "#": 1, "?": 2, "S": 3, "T": 4}
    with open(path, "r", encoding="utf-8-sig") as f:
        xs = [[char_id[c] for c in line.strip()] for line in f if line.strip()]
    return np.asarray(xs, dtype=np.float32)

def read_y(path: str) -> np.ndarray:
    return np.loadtxt(path, delimiter=",", dtype=np.float32, encoding="utf-8-sig")

# ===== 结果写出 =====
def write_result(path: Path, pred: np.ndarray) -> None:
    np.savetxt(path, pred, delimiter=",", fmt="%.6f")

def create_graph(grid):
    g = networkx.Graph()
    dir = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    for x in range(30):
        for y in range(30):
            for dx, dy in dir:
                nx, ny = x + dx, y + dy
                if 0 <= nx < 30 and 0 <= ny < 30 and not grid[x, y] and not grid[nx, ny]:
                    g.add_edge((x, y), (nx, ny))
    return g

def process_each(grid, s, t):
    structure = np.array([[0, 1, 0],
                          [1, 1, 1],
                          [0, 1, 0]])
    labeled_array, nc = label(~grid, structure) # 弱联通
    ns = (labeled_array == labeled_array[s[0], s[1]]).sum() # 起点连通块大小
    nt = (labeled_array == labeled_array[t[0], t[1]]).sum() # 终点连通块大小
    
    g = create_graph(grid)
    dis = networkx.single_source_shortest_path_length(g, s) if s in g.nodes else dict()
    dst = dis[t] if t in dis.keys() else int(1e9) # 最短路

    n = g.number_of_nodes()
    m = g.number_of_edges()
    md = m * 2 / n # 边密度

    od = sum(1 for node, degree in g.degree() if degree == 1) # another kind of 密度
    return nc, ns, nt, dst, n, m, md, od

def extract_features(grid):
    grid = grid.reshape(30, 30)
    features = []
    s = tuple(np.argwhere(grid == 3)[0]) # start point \in Z^2
    t = tuple(np.argwhere(grid == 4)[0]) # terminal
    
    features.extend([(grid == 0).sum(), (grid == 1).sum(), (grid == 2).sum()]) # 每类点数量
    features.extend(process_each(grid == 1, s, t))
    features.extend(process_each((grid == 1) | (grid == 2), s, t))

    xx, yy = np.meshgrid(np.arange(30), np.arange(30), indexing="ij")
    cnt = np.zeros((2, 11, 3))

    for i in range(11):
        for idx, (x, y) in [(0, s), (1, t)]: # enumerate(s,t)
            mdis = np.abs(xx - x) + np.abs(yy - y)
            mask = mdis <= i # L1 distance circle
            for gird_type in range(3):
                cnt[idx, i, gird_type] = ((grid == gird_type) * mask).sum()
    cnt = cnt[:, 1:, :]
    features.extend(cnt.flatten())
    return np.asarray(features)

# ===== 自定义随机森林估计器 =====
class MyEstimator(BaseEstimator, RegressorMixin):
    def __init__(self):
        self.model = MultiOutputRegressor(RandomForestRegressor(n_estimators=100, random_state=seed))

    def fit(self, X, y):
        feat = [extract_features(x) for x in X]
        self.model.fit(feat, np.log(y))
        return self

    def predict(self, X):
        feat = [extract_features(x) for x in X]
        pred = self.model.predict(feat)
        pred = np.exp(pred)
        return pred

# ===== 模型训练 =====
def train_model(train_x_path: str, train_y_path: str) -> MyEstimator:
    x_train = encode_lines(train_x_path)
    y_train = read_y(train_y_path)

    if USE_VALIDATION:
        # 划分训练集和验证集
        x_train, x_val, y_train, y_val = train_test_split(x_train, y_train, test_size=0.2, random_state=seed)
    else:
        x_val, y_val = None, None

    model = MyEstimator()
    model.fit(x_train, y_train)

    # 如果使用验证集，计算验证集得分
    if USE_VALIDATION:
        val_pred = model.predict(x_val)
        val_score = score(y_val, val_pred)
        print(f"Validation Score: {val_score:.4f}")

    return model

# ===== 预测 =====
def predict(model: MyEstimator, test_x_path: str) -> np.ndarray:
    x_test = encode_lines(test_x_path)
    return model.predict(x_test)

def score(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """
    计算预测值的得分。
    
    参数:
    y_true: 真实值，形状为 (n_samples, n_targets)
    y_pred: 预测值，形状为 (n_samples, n_targets)
    
    返回:
    总得分，范围在 0 到 1 之间。
    """
    n_targets = y_true.shape[1]
    total_score = 0.0

    for i in range(n_targets):
        true_values = y_true[:, i]
        pred_values = y_pred[:, i]

        # 计算绝对百分比误差 (APE)
        ape = np.abs(pred_values - true_values) / np.abs(true_values)

        # 计算 MAPE
        mape = np.mean(ape)

        # 计算 Top 10% 平均百分比误差 (Max10PE)
        k = int(np.ceil(0.1 * len(ape)))  # 取前 10% 的样本数量
        max10pe = np.mean(np.sort(ape)[-k:]) if k > 0 else 0.0

        # 计算单个预测值的小分
        score_i = 0.2 * np.exp(-mape) + 0.05 * np.exp(-max10pe)
        total_score += score_i

    return total_score

# ===== 主流程 =====
TRAIN_PATH = "/bohr/train-abk9/v1/" if ONLINE else "C:/Users/Administrator/Documents/NOAI/T4/train_v1/"

# 训练集
train_x_path = TRAIN_PATH + "train_data.csv"
train_y_path = TRAIN_PATH + "train_answer.csv"

out_path = Path("result.csv")

# 训练模型
model = train_model(train_x_path, train_y_path)

In [ ]:
if os.environ.get("DATA_PATH"):
    DATA_PATH = os.environ.get("DATA_PATH") + "/"  # 测试集路径
else:
    DATA_PATH = "/bohr/mazeval-7zx2/v1/"  # 本地测试回退

# 测试集
testA_path = DATA_PATH + "val_data.csv"
testB_path = DATA_PATH + "test_data.csv"

# 分别预测
pred_A = predict(model, testA_path)
pred_B = predict(model, testB_path)

# 合并预测结果
submissionA = pd.DataFrame(pred_A)
submissionA.to_csv("./submission_val.csv", index=False, header=False)

submissionB = pd.DataFrame(pred_B)
submissionB.to_csv("./submission_test.csv", index=False, header=False)

files_to_zip = ['./submission_val.csv', './submission_test.csv']
zip_filename = 'submission.zip'

with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for file in files_to_zip:
        zipf.write(file, os.path.basename(file))

print(f'{zip_filename} is created successfully!')